In [1]:
# ============================================================
#  9회 지방선거 시도지사 당선인 예측
#  필요한 파일:
#    - results_7th.csv : 7회(2018) 선거 결과
#    - results_8th.csv : 8회(2022) 선거 결과
#    - approve_all_regions_cleaned.csv : 9회 여론조사
#    - candidate_info.csv : 9회 후보 목록 (정당, 출마여부)
#    - data_main_regionTypeMap.csv : 지역별 여론조사 유형(다자/양자)
# ============================================================

In [1]:
import ast
import torch
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_score

In [2]:
# ------------------------------------------------------------
# 0. 한글 폰트 설정
# ------------------------------------------------------------
import platform

def set_korean_font():
    if platform.system() == 'Darwin':        # macOS
        plt.rcParams['font.family'] = 'AppleGothic'

    elif platform.system() == 'Windows':     # Windows
        plt.rcParams['font.family'] = 'Malgun Gothic'

    else:                                    # Linux (서버/Colab 등)
        # 설치된 한글 폰트 자동 탐색
        korean_fonts = [f for f in fm.findSystemFonts()
                        if any(k in f for k in ['Nanum', 'Gothic', 'CJK', 'nanum'])]
        if korean_fonts:
            fm.fontManager.addfont(korean_fonts[0])
            plt.rcParams['font.family'] = fm.FontProperties(fname=korean_fonts[0]).get_name()
        else:
            print("한글 폰트를 찾지 못했습니다.")

    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()

In [3]:
# ------------------------------------------------------------
# 1. 파일 불러오기
# ------------------------------------------------------------
from pathlib import Path

# 노트북 파일 기준으로 경로 자동 설정
# BASE_DIR = Path(__file__).parent  # .py 파일
BASE_DIR = Path().resolve().parent.parent  # .ipynb 파일

DATA = BASE_DIR / 'data' / 'processed'
RAW  = BASE_DIR / 'data' / 'raw'
 
# 7회·8회 선거 결과 
df7  = pd.read_csv(DATA / 'results_7th.csv')
df8  = pd.read_csv(DATA / 'results_8th.csv')
 
# 9회 여론조사
poll = pd.read_csv(DATA / 'approve_all_regions_cleaned.csv')
poll['date'] = pd.to_datetime(poll['date'])
 
# 9회 후보 정보(정당, 출마 여부)
cand_raw = pd.read_csv(DATA / 'candidate_info.csv')
cand_raw['cands'] = cand_raw['value'].apply(ast.literal_eval)
 
# 지역별 여론조사 유형(다자/양자)
type_map = pd.read_csv(DATA / 'data_main_regionTypeMap.csv')
type_map = dict(zip(type_map['key'], type_map['value']))
 
print(f"7회 후보 수 : {len(df7)}명, 8회 후보 수 : {len(df8)}명")
print(f"여론조사 행 수 : {len(poll)}행")

7회 후보 수 : 71명, 8회 후보 수 : 54명
여론조사 행 수 : 1503행


In [4]:
# ------------------------------------------------------------
# 2. 후보 → 정당 매핑 딕셔너리 만들기
# candidate_info.csv 에서 isRUN=True인 후보만 사용
# ------------------------------------------------------------
name_to_party = {}   # 예: {'정원오': '더불어민주당', '오세훈': '국민의힘', ...}
active_by_region = {}  # 예: {'서울특별시': ['정원오', '오세훈', ...], ...}
 
for _, row in cand_raw.iterrows():
    region_name = row['key']
    candidates  = row['cands']
 
    # isRUN=False(사퇴/미출마) 제외
    active = [c for c in candidates if c.get('isRUN', True)]
 
    active_by_region[region_name] = [c['name'] for c in active]
    for c in active:
        name_to_party[c['name']] = c['party']
 
# 지역 이름 매핑 : 한글 시도명 ↔ 여론조사 파일의 영문 지역명
kr_to_eng = {
    '서울특별시':     'Seoul',
    '부산광역시':     'Busan',
    '대구광역시':     'Daegu',
    '인천광역시':     'Incheon',
    '대전광역시':     'Daejeon',
    '울산광역시':     'Ulsan',
    '세종특별자치시':  'Sejong',
    '경기도':        'Gyeonggi',
    '강원특별자치도':  'Gangwon',
    '충청북도':      'Chungbuk',
    '충청남도':      'Chungnam',
    '전북특별자치도':  'Jeonbuk',
    '경상북도':      'Gyeongbuk',
    '경상남도':      'Gyeongnam',
    '제주특별자치도':  'Jeju',
    '전남광주통합특별시': None,   # 여론조사 없음
}

print(f"총 {len(name_to_party)}명의 후보 정당 매핑")

총 62명의 후보 정당 매핑


In [5]:
# ------------------------------------------------------------
# 3. 훈련 데이터 만들기
# 각 선거의 후보별로 "이 후보가 당선될까?"를 예측하기 위한 숫자 feature 계산
# ------------------------------------------------------------
def make_features(df):
    """
    입력: sido / name / party / vote_rate / winner 컬럼을 가진 DataFrame
    출력: feature 컬럼이 추가된 DataFrame
 
    만들 feature:
      - rank         : 같은 지역 내 득표율 순위 (1위=1)
      - gap_to_1st   : 1위와의 득표율 차이 (1위는 0, 나머지는 음수)
      - gap_top2     : 1위-2위 격차 (경쟁이 치열할수록 작음)
      - n_cands      : 해당 지역 후보 수
      - is_minjoo    : 더불어민주당 후보이면 1, 아니면 0
      - is_gukmin    : 국민의힘 후보이면 1, 아니면 0
    """
    rows = []
 
    for sido, group in df.groupby('sido'):
        # 득표율 내림차순 정렬
        group = group.sort_values('vote_rate', ascending=False).reset_index(drop=True)
 
        top1_rate = group.iloc[0]['vote_rate']
        top2_rate = group.iloc[1]['vote_rate'] if len(group) > 1 else 0
        gap_top2  = top1_rate - top2_rate
        n_cands   = len(group)
 
        for rank, row in enumerate(group.itertuples(), start=1):
            rows.append({
                # 원본 정보
                'sido':       sido,
                'name':       row.name,
                'party':      row.party,
                'vote_rate':  row.vote_rate,
                'winner':     row.winner,
                # feature
                'rank':       rank,
                'gap_to_1st': row.vote_rate - top1_rate,  # 1위는 0, 나머지 음수
                'gap_top2':   gap_top2,
                'n_cands':    n_cands,
                'is_minjoo':  int('민주' in row.party),
                'is_gukmin':  int('국민의힘' in row.party or '자유한국당' in row.party),
            })
 
    return pd.DataFrame(rows)

train7 = make_features(df7)
train8 = make_features(df8)
train  = pd.concat([train7, train8], ignore_index=True)
 
print(f"총 {len(train)}행 (당선={train['winner'].sum()}명, 낙선={(train['winner']==0).sum()}명)")

총 125행 (당선=34명, 낙선=91명)


In [6]:
# ------------------------------------------------------------
# 4. 모델 학습
# Logistic Regression : 당선 확률을 0~1 사이 숫자로 출력
# 소표본(34 레이스)이라 단순 모델이 더 나음
# ------------------------------------------------------------
FEATURES = ['vote_rate', 'rank', 'gap_to_1st', 'gap_top2', 'n_cands', 'is_minjoo', 'is_gukmin']
 
X = train[FEATURES].values
y = train['winner'].values
 
# 스케일링(feature 크기 맞춰줌)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
# 모델
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
 
# 교차검증 : LOO(Leave-One-Out) 사용
loo_scores = cross_val_score(model, X_scaled, y, cv=LeaveOneOut(), scoring='accuracy')
print(f"LOO 교차검증 정확도 : {loo_scores.mean():.1%}")
 
# 전체 데이터로 최종 학습
model.fit(X_scaled, y)

LOO 교차검증 정확도 : 99.2%


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [7]:
# ------------------------------------------------------------
# 5. 9회 여론조사 데이터 준비
# 각 지역의 가장 최신 여론조사 가져오기
# ------------------------------------------------------------
def get_latest_poll(eng_region, poll_type):
    """
    특정 지역·유형의 최신 여론조사 반환
    '없음' 응답은 후보가 아니므로 제외
    """
    filtered = poll[
        (poll['region'] == eng_region) &
        (poll['type']   == poll_type)  &
        (poll['name']   != '없음')
    ]
    if filtered.empty:
        return None
 
    latest_date = filtered['date'].max()
    return filtered[filtered['date'] == latest_date].copy()
 
 
# 9회 예측용 feature 생성
pred_rows = []
 
for kr_name, active_names in active_by_region.items():
    eng_name  = kr_to_eng.get(kr_name)
    poll_type = type_map.get(kr_name, '다자')
 
    # 여론조사 없는 지역 건너뜀
    if eng_name is None:
        print(f">>>>> {kr_name}: 여론조사 없음 → 건너뜀")
        continue
 
    # 최신 여론조사 가져오기
    latest = get_latest_poll(eng_name, poll_type)
 
    # 해당 유형 없으면 '다자'로 재시도
    if latest is None:
        latest = get_latest_poll(eng_name, '다자')
    if latest is None:
        print(f">>>>> {kr_name}: 여론조사 데이터 없음 → 건너뜀")
        continue
 
    # isRUN=True 후보만 필터
    latest = latest[latest['name'].isin(active_names)]
    if latest.empty:
        print(f">>>>> {kr_name}: 출마 후보 여론조사 없음 → 건너뜀")
        continue
 
    # feature 계산(make_features와 동일 방식)
    latest = latest.sort_values('mean', ascending=False).reset_index(drop=True)
    top1_rate = latest.iloc[0]['mean']
    top2_rate = latest.iloc[1]['mean'] if len(latest) > 1 else 0
    gap_top2  = top1_rate - top2_rate
    n_cands   = len(latest)
 
    for rank, row in enumerate(latest.itertuples(), start=1):
        party = name_to_party.get(row.name, '기타')
        pred_rows.append({
            # 정보
            'region_kr':  kr_name,
            'region_eng': eng_name,
            'name':       row.name,
            'party':      party,
            'poll_mean':  row.mean,
            'poll_date':  str(row.date.date()),
            'poll_type':  poll_type,
            # feature(훈련 때와 동일 이름)
            'vote_rate':  row.mean,
            'rank':       rank,
            'gap_to_1st': row.mean - top1_rate,
            'gap_top2':   gap_top2,
            'n_cands':    n_cands,
            'is_minjoo':  int('더불어민주당' in party),
            'is_gukmin':  int('국민의힘' in party),
        })
 
pred = pd.DataFrame(pred_rows)
print(f"9회 예측 데이터 : {pred['region_kr'].nunique()}개 지역, {len(pred)}명")

>>>>> 전남광주통합특별시: 여론조사 없음 → 건너뜀
9회 예측 데이터 : 15개 지역, 44명


In [8]:
# ------------------------------------------------------------
# 6. 당선 확률 예측
# ------------------------------------------------------------
X_pred = pred[FEATURES].values
X_pred_scaled = scaler.transform(X_pred)
 
pred['win_prob'] = model.predict_proba(X_pred_scaled)[:, 1]
 
# 각 지역에서 win_prob 가장 높은 후보 = 예측 당선자
pred['predicted_winner'] = (
    pred['win_prob'] == pred.groupby('region_kr')['win_prob'].transform('max')
).astype(int)
 
# 경쟁도 라벨
def label_competition(gap):
    if gap > 20: return '압승 예상'
    if gap > 10: return '우세'
    if gap >  5: return '경쟁'
    return '초경쟁'
 
pred['경쟁도'] = pred['gap_top2'].apply(label_competition)

In [9]:
# ------------------------------------------------------------
# 7. 결과 출력
# ------------------------------------------------------------
winners = pred[pred['predicted_winner'] == 1].sort_values('region_kr').copy()
winners['당선확률'] = (winners['win_prob'] * 100).round(1).astype(str) + '%'
winners['여론조사'] = winners['poll_mean'].astype(str) + '%'
winners['격차']    = winners['gap_top2'].round(1).astype(str) + '%p'
 
print("\n" + "="*65)
print("9회 지방선거 시도지사 예측 결과")
print("="*65)
print(f"{'지역':<12} {'후보':<6} {'정당':<10} {'여론조사':<8} {'당선확률':<8} {'격차':<7} {'경쟁도'}")
print("-"*65)
for _, r in winners.iterrows():
    print(f"{r['region_kr']:<12} {r['name']:<6} {r['party']:<10} "
          f"{r['여론조사']:<8} {r['당선확률']:<8} {r['격차']:<7} {r['경쟁도']}")
 
print("\n정당별 예측 당선 수:")
print(winners['party'].value_counts().to_string())


9회 지방선거 시도지사 예측 결과
지역           후보     정당         여론조사     당선확률     격차      경쟁도
-----------------------------------------------------------------
강원특별자치도      우상호    더불어민주당     48.0%    68.4%    6.8%p   경쟁
경기도          추미애    더불어민주당     53.1%    83.3%    20.7%p  압승 예상
경상남도         김경수    더불어민주당     43.9%    59.4%    0.2%p   초경쟁
경상북도         이철우    국민의힘       55.7%    88.9%    24.8%p  압승 예상
대구광역시        추경호    국민의힘       45.2%    70.5%    3.8%p   초경쟁
대전광역시        허태정    더불어민주당     47.4%    71.2%    11.5%p  우세
부산광역시        전재수    더불어민주당     46.5%    68.5%    9.0%p   경쟁
서울특별시        정원오    더불어민주당     44.1%    63.4%    3.9%p   초경쟁
세종특별자치시      조상호    더불어민주당     52.7%    82.9%    23.4%p  압승 예상
울산광역시        김상욱    더불어민주당     36.4%    50.2%    5.3%p   경쟁
인천광역시        박찬대    더불어민주당     52.2%    77.8%    14.1%p  우세
전북특별자치도      김관영    무소속        44.3%    73.0%    3.0%p   초경쟁
제주특별자치도      위성곤    더불어민주당     58.8%    91.3%    37.3%p  압승 예상
충청남도         박수현    더불어민주당     47.5%    69.4%    10.3%p  

In [10]:
# ------------------------------------------------------------
# 8. 시각화
# ------------------------------------------------------------
PARTY_COLOR = {
    '더불어민주당': '#1259b6',
    '국민의힘':    '#e61e2b',
    '개혁신당':    '#ff7f00',
    '진보당':      '#8B0000',
    '정의당':      '#ffcc00',
    '무소속':      '#888888',
    '기타':        '#cccccc',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('9회 지방선거 시도지사 당선 예측', fontsize=16, fontweight='bold', y=1.01)
 
# ── 왼쪽 : 지역별 당선 확률 ────────────────────
ax = axes[0]
ws = winners.sort_values('win_prob')
bar_colors = [PARTY_COLOR.get(p, '#ccc') for p in ws['party']]
bars = ax.barh(ws['region_kr'], ws['win_prob'] * 100, color=bar_colors, edgecolor='white')
 
for bar, row in zip(bars, ws.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{row.name}  {row.win_prob*100:.0f}%", va='center', fontsize=9)
 
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('당선 확률 (%)')
ax.set_xlim(0, 115)
ax.set_title('① 지역별 예측 당선자 당선 확률', fontsize=12, fontweight='bold')
used_parties = ws['party'].unique()
handles = [Patch(color=PARTY_COLOR.get(p, '#ccc'), label=p) for p in used_parties]
ax.legend(handles=handles, fontsize=8, loc='lower right')
 
 
# ── 오른쪽 : 경합도 (1위-2위 격차) ─────────────
ax2 = axes[1]
gap_data = pred[pred['rank'] == 1].sort_values('gap_top2')
gap_colors = ['#e74c3c' if g <= 5 else '#f39c12' if g <= 10 else '#27ae60'
              for g in gap_data['gap_top2']]
ax2.barh(gap_data['region_kr'], gap_data['gap_top2'], color=gap_colors, edgecolor='white')
 
for i, (_, row) in enumerate(gap_data.iterrows()):
    ax2.text(row['gap_top2'] + 0.3, i, f"{row['gap_top2']:.1f}%p", va='center', fontsize=9)
 
ax2.axvline(5,  color='#e74c3c', linestyle='--', lw=1, label='5%p 이하: 초경쟁')
ax2.axvline(10, color='#f39c12', linestyle='--', lw=1, label='10%p 이하: 경쟁')
ax2.set_xlabel('1위-2위 격차 (%p)')
ax2.set_title('② 지역별 경합도', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig(DATA / 'election_result.png', dpi=150, bbox_inches='tight')
print(f"차트 저장: {DATA}election_result.png")

차트 저장: /Users/jeonseoyeon/Documents/SKN31_2nd/SKN31-2nd-1Team/data/processedelection_result.png


In [11]:
# ============================================================
# 추가 차트 : 정당 지지율 추이(꺾은선)
# ============================================================
party = pd.read_csv(DATA / 'party_all.csv')
party['date'] = pd.to_datetime(party['date'])

fig, ax = plt.subplots(figsize=(14, 6))
valid = party.dropna(subset=['더불어민주당_mean', '국민의힘_mean'])
 
PARTY_TREND = {
    '더불어민주당': ('#1259b6', '더불어민주당_mean', '더불어민주당_lower', '더불어민주당_upper'),
    '국민의힘':    ('#e61e2b', '국민의힘_mean',    '국민의힘_lower',    '국민의힘_upper'),
    '조국혁신당':  ('#6a0dad', '조국혁신당_mean',  None,               None),
    '개혁신당':    ('#ff7f00', '개혁신당_mean',    None,               None),
}
 
for party_name, (color, mean_col, lower_col, upper_col) in PARTY_TREND.items():
    sub = valid.dropna(subset=[mean_col])
    ax.plot(sub['date'], sub[mean_col], color=color, lw=2,
            label=party_name, marker='o', markersize=2)
    if lower_col and upper_col:
        ax.fill_between(sub['date'], sub[lower_col], sub[upper_col],
                        color=color, alpha=0.08)
 
ax.set_xlabel('날짜', fontsize=11)
ax.set_ylabel('지지율 (%)', fontsize=11)
ax.set_title('차트 5. 전국 정당 지지율 추이', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.2)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
 
plt.tight_layout()
plt.savefig(DATA / 'party_trend.png', dpi=150, bbox_inches='tight')
plt.close()

In [12]:
# ============================================================
# 추가 차트 : 지역별 여론조사 추이(꺾은선, 3x5 서브플롯)
# ============================================================
regions = sorted(poll['region'].unique())
fig, axes = plt.subplots(3, 5, figsize=(22, 13), sharey=False)
fig.suptitle('차트 6. 지역별 후보 여론조사 추이', fontsize=14, fontweight='bold', y=1.01)
 
ENG_TO_KR = {
    'Seoul':'서울', 'Busan':'부산', 'Daegu':'대구', 'Incheon':'인천',
    'Daejeon':'대전', 'Ulsan':'울산', 'Sejong':'세종', 'Gyeonggi':'경기',
    'Gangwon':'강원', 'Chungbuk':'충북', 'Chungnam':'충남',
    'Jeonbuk':'전북', 'Gyeongbuk':'경북', 'Gyeongnam':'경남', 'Jeju':'제주',
}
 
# 후보→정당 매핑(pred 데이터 활용)
name_to_party = dict(zip(pred['name'], pred['party']))
 
poll_f = poll[(poll['type'].isin(['다자','양자'])) & (poll['name'] != '없음')]
 
for ax_i, (ax, region) in enumerate(zip(axes.flatten(), regions)):
    sub = poll_f[poll_f['region'] == region]
    for cand_name, grp in sub.groupby('name'):
        grp = grp.sort_values('date')
        party_name = name_to_party.get(cand_name, '기타')
        color = PARTY_COLOR.get(party_name, '#aaaaaa')
        ax.plot(grp['date'], grp['mean'], color=color, lw=1.8,
                label=cand_name, marker='o', markersize=3)
        ax.fill_between(grp['date'], grp['lower'], grp['upper'],
                        color=color, alpha=0.08)
 
    ax.set_title(ENG_TO_KR.get(region, region), fontsize=11, fontweight='bold')
    ax.set_ylim(0, 75)
    ax.grid(True, alpha=0.2)
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=7, loc='upper left')
 
plt.tight_layout()
plt.savefig(DATA / 'poll_trend.png', dpi=150, bbox_inches='tight')
plt.close()

In [13]:
# ============================================================
# 추가 차트 : 후보별 여론조사 신뢰구간
# ============================================================
poll_f = poll[(poll['type'].isin(['다자','양자'])) & (poll['name'] != '없음')]
 
regions_ordered = ['Seoul','Gyeonggi','Incheon','Busan','Daegu','Daejeon',
                   'Ulsan','Sejong','Gangwon','Chungbuk','Chungnam',
                   'Jeonbuk','Gyeongbuk','Gyeongnam','Jeju']
ENG_TO_KR = {
    'Seoul':'서울','Gyeonggi':'경기','Incheon':'인천','Busan':'부산',
    'Daegu':'대구','Daejeon':'대전','Ulsan':'울산','Sejong':'세종',
    'Gangwon':'강원','Chungbuk':'충북','Chungnam':'충남','Jeonbuk':'전북',
    'Gyeongbuk':'경북','Gyeongnam':'경남','Jeju':'제주',
}
name_to_party = dict(zip(pred['name'], pred['party']))
 
fig, axes = plt.subplots(3, 5, figsize=(22, 14))
fig.suptitle('차트 11. 지역별 후보 여론조사 신뢰구간 (최신 조사 기준)', fontsize=14, fontweight='bold')
 
for ax, region in zip(axes.flatten(), regions_ordered):
    sub = poll_f[poll_f['region'] == region]
    latest_date = sub['date'].max()
    latest = sub[sub['date'] == latest_date].copy()
    latest = latest[latest['name'] != '없음'].sort_values('mean', ascending=True)
    colors = [PARTY_COLOR.get(name_to_party.get(n, '기타'), '#ccc') for n in latest['name']]
 
    # 신뢰구간 표시(가로 막대 + 에러바)
    y = np.arange(len(latest))
    bars = ax.barh(y, latest['mean'].values, color=colors,
                   edgecolor='white', height=0.6, alpha=0.85)
    ax.errorbar(latest['mean'].values, y,
                xerr=[latest['mean'].values - latest['lower'].values,
                      latest['upper'].values - latest['mean'].values],
                fmt='none', color='#333', capsize=4, lw=1.5, zorder=4)
 
    # 수치 표시
    for bar, row in zip(bars, latest.itertuples()):
        ax.text(row.upper + 0.5, bar.get_y() + bar.get_height()/2,
                f"{row.mean:.1f}%", va='center', fontsize=7.5, color='#333')
 
    ax.set_yticks(y)
    ax.set_yticklabels(latest['name'].values, fontsize=8)
    ax.set_xlim(0, 70)
    ax.set_title(f"{ENG_TO_KR.get(region, region)}  ({str(latest_date.date())})",
                 fontsize=9.5, fontweight='bold')
    ax.axvline(50, color='gray', lw=0.8, linestyle='--', alpha=0.4)
    ax.grid(True, axis='x', alpha=0.2)
 
plt.tight_layout()
plt.savefig(DATA / 'confidence_interval.png', dpi=150, bbox_inches='tight')
plt.close()

In [15]:
# ------------------------------------------------------------
# 9. CSV 저장
# ------------------------------------------------------------
save_cols = ['region_kr', 'name', 'party', 'poll_mean', 'poll_date',
             'poll_type', 'win_prob', 'predicted_winner', '경쟁도']
pred[save_cols].sort_values(['region_kr', 'win_prob'], ascending=[True, False]) \
    .to_csv(DATA / 'prediction_9th.csv', index=False, encoding='utf-8-sig')
 
print(f"결과 저장: {DATA}predict_9th_cp.csv")

결과 저장: /Users/jeonseoyeon/Documents/SKN31_2nd/SKN31-2nd-1Team/data/processedpredict_9th_cp.csv


In [16]:
# ------------------------------------------------------------
# 모델 저장 (최종 완성 모델)
# ------------------------------------------------------------
MODELS_DIR = BASE_DIR / 'ML' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 학습 완료된 모델 저장
saved_model = {
    'model':          model,       # 학습 완료된 Logistic Regression
    'scaler':         scaler,      # 함께 저장 필수 (예측 시 필요)
    'features':       FEATURES,    # 피처 순서 기록
}

torch.save(saved_model, MODELS_DIR / 'logistic_model.pth')
print(f"모델 저장: {MODELS_DIR / 'logistic_model.pth'}")

모델 저장: /Users/jeonseoyeon/Documents/SKN31_2nd/SKN31-2nd-1Team/ML/models/logistic_model.pth


In [17]:
# ------------------------------------------------------------
# 모델 불러와서 바로 예측
# ------------------------------------------------------------
saved = torch.load(MODELS_DIR / 'logistic_model.pth', weights_only=False)

model_loaded  = saved['model']
scaler_loaded = saved['scaler']

# pred DataFrame에서 피처 추출 → 스케일링 → 예측
X_pred_scaled = scaler_loaded.transform(pred[saved['features']])
pred['win_prob'] = model_loaded.predict_proba(X_pred_scaled)[:, 1]

/Users/jeonseoyeon/Documents/SKN31_2nd/SKN31-2nd-1Team/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
